In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report

from sklearn.ensemble import RandomForestClassifier
import joblib

In [3]:
import os, glob

# List dataset folders
print("Folders under /kaggle/input:")
print(os.listdir("/kaggle/input"))

# Try to locate the folder that matches your dataset (adjust if needed)
candidates = [p for p in glob.glob("/kaggle/input/*") if os.path.isdir(p)]
for p in candidates:
    if "github" in p.lower() or "repo" in p.lower() or "2026" in p.lower():
        print("Candidate:", p)
        print("Files:", os.listdir(p)[:50])

Folders under /kaggle/input:
['datasets']


In [4]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for f in files:
        print("  -", f)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/elvisbui
/kaggle/input/datasets/elvisbui/top-github-repositories-2026
  - top_github_repos_2026.csv


In [5]:
import pandas as pd

DATA_PATH = "/kaggle/input/datasets/elvisbui/top-github-repositories-2026/top_github_repos_2026.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head(5))

Shape: (627, 17)
Columns: ['name', 'description', 'language', 'stars', 'forks', 'open_issues', 'watchers', 'created_at', 'updated_at', 'topics', 'license', 'is_fork', 'has_wiki', 'archived', 'size_kb', 'default_branch', 'homepage']


,name,description,language,stars,forks,open_issues,watchers,created_at,updated_at,topics,license,is_fork,has_wiki,archived,size_kb,default_branch,homepage
0,codecrafters-io/build-your-own-x,Master programming by recreating your favorite...,Markdown,487662,45893,462,487662,2018-05-09,2026-04-09,awesome-list|free|programming|tutorial-code|tu...,NaN,False,False,False,1201,master,https://codecrafters.io
1,sindresorhus/awesome,😎 Awesome lists about all kinds of interesting...,NaN,453320,34056,59,453320,2014-07-11,2026-04-09,awesome|awesome-list|lists|resources|unicorns,CC0-1.0,False,False,False,1535,main,NaN
2,freeCodeCamp/freeCodeCamp,freeCodeCamp.org's open-source codebase and cu...,TypeScript,442235,44192,200,442235,2014-12-24,2026-04-09,careers|certification|community|curriculum|d3|...,BSD-3-Clause,False,False,False,559313,main,https://contribute.freecodecamp.org
3,public-apis/public-apis,A collective list of free APIs,Python,420283,45730,1216,420283,2016-03-20,2026-04-09,api|apis|dataset|development|free|list|lists|o...,MIT,False,False,False,4962,master,https://APILayer.com/?utm_source=Github&utm_me...
4,EbookFoundation/free-programming-books,:books: Freely available programming books,Python,385209,66095,79,385209,2013-10-11,2026-04-09,books|education|hacktoberfest|list|resource,CC-BY-4.0,False,False,False,21200,main,https://ebookfoundation.github.io/free-program...


In [13]:
import numpy as np

df.columns = [c.strip().lower() for c in df.columns]

# Common renames
rename_map = {
    "stargazers": "stars",
    "stargazers_count": "stars",
    "star_count": "stars",
    "forks_count": "forks",
    "open_issues_count": "open_issues",
}
df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})

# Ensure numeric stars
if "stars" not in df.columns:
    raise ValueError(f"No stars column found. Columns are: {df.columns.tolist()}")

df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
df = df.dropna(subset=["stars"]).copy()

# Classification target: top 20% are "high star"
thr = float(df["stars"].quantile(0.80))
df["high_star_repo"] = (df["stars"] >= thr).astype(int)

print("High-star threshold (80th percentile):", thr)
print(df["high_star_repo"].value_counts(normalize=True))

High-star threshold (80th percentile): 83223.6
high_star_repo
0    0.799043
1    0.200957
Name: proportion, dtype: float64


In [14]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier

# Pick available columns
candidate_features = ["forks", "open_issues", "language", "license", "topics"]
feature_cols = [c for c in candidate_features if c in df.columns]

X = df[feature_cols].copy()
y = df["high_star_repo"].copy()

num_cols = [c for c in feature_cols if c in ["forks", "open_issues"]]
cat_cols = [c for c in feature_cols if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

clf = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(
        n_estimators=500,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample"
    ))
])

clf.fit(X_train, y_train)

proba = clf.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Features used:", feature_cols)
print("ROC AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))

Features used: ['forks', 'open_issues', 'language', 'license', 'topics']
ROC AUC: 0.8972277227722774
              precision    recall  f1-score   support

           0       0.89      0.99      0.94       101
           1       0.93      0.52      0.67        25

    accuracy                           0.90       126
   macro avg       0.91      0.76      0.80       126
weighted avg       0.90      0.90      0.88       126



In [15]:
import joblib, os

artifact = {
    "pipeline": clf,
    "star_threshold": thr,
    "feature_cols": feature_cols
}
OUT_PATH = "github_repo_success_model.joblib"
joblib.dump(artifact, OUT_PATH)

print("Saved:", OUT_PATH)
print("Size bytes:", os.path.getsize(OUT_PATH))

Saved: github_repo_success_model.joblib
Size bytes: 10614019


In [16]:
print(df.columns.tolist())

['name', 'description', 'language', 'stars', 'forks', 'open_issues', 'watchers', 'created_at', 'updated_at', 'topics', 'license', 'is_fork', 'has_wiki', 'archived', 'size_kb', 'default_branch', 'homepage', 'high_star_repo']


In [18]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import joblib

DATA_PATH = "/kaggle/input/datasets/elvisbui/top-github-repositories-2026/top_github_repos_2026.csv"
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().lower() for c in df.columns]

# -----------------------------
# 1) Target: high_star_repo
# -----------------------------
df["stars"] = pd.to_numeric(df["stars"], errors="coerce")
df = df.dropna(subset=["stars"]).copy()

# Top 20% stars => high_star_repo
star_threshold = float(df["stars"].quantile(0.80))
df["high_star_repo"] = (df["stars"] >= star_threshold).astype(int)

print("Star threshold (80th percentile):", star_threshold)
print(df["high_star_repo"].value_counts(normalize=True))

# -----------------------------
# 2) Feature engineering
# -----------------------------
for c in ["created_at", "updated_at"]:
    df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)

ASOF = pd.Timestamp("2026-05-07", tz="UTC")
df["repo_age_days"] = (ASOF - df["created_at"]).dt.days
df["days_since_update"] = (ASOF - df["updated_at"]).dt.days

def normalize_topics(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    s = s.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    s = s.replace(";", ",").replace("|", ",")
    parts = [p.strip().lower().replace(" ", "-") for p in s.split(",")]
    parts = [p for p in parts if p]
    seen = set()
    out = []
    for p in parts:
        if p not in seen:
            out.append(p)
            seen.add(p)
    return ",".join(out)

if "topics" in df.columns:
    df["topics"] = df["topics"].apply(normalize_topics)

# -----------------------------
# 3) Train
# -----------------------------
feature_cols = [
    "forks", "open_issues", "watchers",
    "language", "license", "topics",
    "is_fork", "has_wiki", "archived",
    "size_kb", "default_branch",
    "repo_age_days", "days_since_update",
]
feature_cols = [c for c in feature_cols if c in df.columns]

X = df[feature_cols].copy()
y = df["high_star_repo"].astype(int).copy()

# Convert booleans consistently (if they are True/False strings)
for c in ["is_fork", "has_wiki", "archived"]:
    if c in X.columns:
        X[c] = X[c].astype(str).str.lower().replace({"true": 1, "false": 0})
        X[c] = pd.to_numeric(X[c], errors="coerce")

numeric_cols = [c for c in feature_cols if c in [
    "forks","open_issues","watchers","size_kb","repo_age_days","days_since_update",
    "is_fork","has_wiki","archived"
]]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols),
    ],
    remainder="drop"
)

model = RandomForestClassifier(
    n_estimators=700,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

pipeline = Pipeline(steps=[("preprocess", preprocess), ("model", model)])
pipeline.fit(X_train, y_train)

proba = pipeline.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print("Features used:", feature_cols)
print("ROC AUC:", roc_auc_score(y_test, proba))
print(classification_report(y_test, pred))

# -----------------------------
# 4) Export for Streamlit
# -----------------------------
artifact = {
    "pipeline": pipeline,
    "feature_cols": feature_cols,
    "asof": str(ASOF),
    "star_threshold": star_threshold,
}
joblib.dump(artifact, "github_repo_success_model.joblib")
print("Saved github_repo_success_model.joblib")

Star threshold (80th percentile): 83223.6
high_star_repo
0    0.799043
1    0.200957
Name: proportion, dtype: float64


/tmp/ipykernel_57/1880269298.py:77: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[c] = X[c].astype(str).str.lower().replace({"true": 1, "false": 0})
/tmp/ipykernel_57/1880269298.py:77: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X[c] = X[c].astype(str).str.lower().replace({"true": 1, "false": 0})
/tmp/ipykernel_57/1880269298.py:77: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to t

Features used: ['forks', 'open_issues', 'watchers', 'language', 'license', 'topics', 'is_fork', 'has_wiki', 'archived', 'size_kb', 'default_branch', 'repo_age_days', 'days_since_update']
ROC AUC: 1.0
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       101
           1       1.00      0.96      0.98        25

    accuracy                           0.99       126
   macro avg       1.00      0.98      0.99       126
weighted avg       0.99      0.99      0.99       126

Saved github_repo_success_model.joblib
